In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

/workspaces/llm-engineering/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.environ.get("AI_API_KEY")
# api_key = os.getenv("GOOGLE_API_KEY")

if api_key:
    api_key = api_key.strip()
    if not api_key:
        print("AI_API_KEY is empty after stripping")
    else:
        print("AI_API_KEY environment variable is set and valid")
else:
    print("AI_API_KEY environment variable not set")

AI_API_KEY environment variable is set and valid


In [3]:
system_prompt = "You are and helpful assistant for a Airline called FlightAI. Give short and concise answers with not more than one sentence. Always be accurate and if you do not know the answer then say so"

In [4]:
client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=api_key
    )

In [5]:
ticket_prices = {"hyd":"2000","mys":"1000","coorg":"1500"}

def get_ticket_price(dest):
    print(f"Called for the destination : {dest}")
    price = ticket_prices.get(dest,"No price details found")
    return f"The flight ticket for the city {dest} is {price}"





In [6]:
price_function = {
    "name":"get_ticket_price",
    "description":"Get the price of a return ticket to a destination city",
    "parameters":{
        "type":"object",
        "properties":{
            "dest": {
                "type": "string",
                "description": "The city for which the customer wants to travel to"
            }
        },
        "required":["dest"],
        "additionalProperties":False
    }
}

In [7]:
tools = [{"type":"function","function":price_function}]

In [8]:
def chat(msgs,history):
    history = [{"role":h["role"],"content":h["content"]} for h in history]
    messages = [{"role":"system","content":system_prompt}]+history+[{"role":"user","content":msgs}]
    response = client.chat.completions.create(model="openai/gpt-oss-20b:free",messages=messages,tools=tools)
    
    if response.choices[0].finish_reason == 'tool_calls':
        message = response.choices[0].message
        tool_response = handle_tool_call(message)
        messages.append(message)
        messages.append(tool_response)
        final_response = client.chat.completions.create(model="openai/gpt-oss-20b:free",messages=messages)
        return final_response.choices[0].message.content
    
    return response.choices[0].message.content

In [9]:
import json
def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('dest')
        price_details = get_ticket_price(city)
        response = {
            "role":"tool",
            "content":price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [10]:
import gradio as gr

gr.ChatInterface(chat).launch()



* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Called for the destination : Mysore


Traceback (most recent call last):
  File "/workspaces/llm-engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 856, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspaces/llm-engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 358, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspaces/llm-engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2179, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspaces/llm-engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1634, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspaces/llm-engineering/.venv/lib/python3.12/site-packages/gradio/utils.py", line 1027, in async_wrapper
   